# Federated K-Means Clustering with Scikit-learn on Iris Dataset

## Federated mini-batch K-means

This example clusters the Iris dataset at three simulated sites. Round zero pools k-means++ seeds to initialize global centers; later rounds combine matching centers using per-center update counts and historical global counts. This running average need not match centralized K-means.

The initial seeds are exact local feature rows. Later centroids can also reveal individual rows, for example in singleton clusters. Use this example only where sharing those values is permitted; it provides no differential privacy or secure aggregation. Labels stay out of training and are used only for the homogeneity metric.

## Install lesson dependencies

First complete the [course environment setup](../../02.0_introduction/introduction.ipynb) to install the matching NVFLARE version and select its Jupyter kernel. The requirements below contain only the additional dependencies for this lesson.

In [ ]:
import os
import sys
from pathlib import Path

# Make notebook shell commands use the selected kernel's environment.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")

lesson = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "convert_kmeans_to_fl.ipynb").is_file())
code_dir = lesson / "code"
print("Example code:", code_dir)
%cd {code_dir}

In [ ]:
%cd {code_dir}
!python -m pip install -r requirements.txt
assert _exit_code == 0, "Command failed; see the output above."

## Prepare Iris data

In [ ]:
%cd {code_dir}
!python prepare_data.py --dataset_name iris --out_path /tmp/nvflare/dataset/sklearn_iris.csv
assert _exit_code == 0, "Command failed; see the output above."

## Define the client and recipe

The Client API receives centers and returns local centers, update counts, and homogeneity. `KMeansFedAvgRecipe` supplies the matching assembler and model persistence. `set_per_site_config` assigns disjoint training ranges. For this small demonstration, all sites validate on the same public Iris dataset. `--split_mode` controls the relative training sizes.

Each site needs at least three training rows to initialize three clusters. `job.py` checks the training ranges before starting or exporting a recipe and reports any undersized sites. For the 150-row Iris dataset, five-client `square` and `exponential` splits are too uneven; use fewer clients or a less imbalanced split such as `uniform` or `linear`.

In [ ]:
print((code_dir / "client.py").read_text())

print((code_dir / "job.py").read_text())

## Run and export the recipe

`SimEnv(log_config="progress")` shows workflow progress and a final run summary. Detailed logs remain in the simulation workspace. Each command below uses this notebook's Python and stops if the command fails.

Run `job.py --export` to export the same recipe without training. The exported job can be submitted to a deployed NVIDIA FLARE system.

In [ ]:
%cd {code_dir}
!python job.py
assert _exit_code == 0, "Command failed; see the output above."

In [ ]:
%cd {code_dir}
!python job.py --export --export-dir /tmp/nvflare/chapter2/exports/kmeans
assert _exit_code == 0, "Command failed; see the output above."

## Results

Each round reports client homogeneity. The server saves the global centers under `/tmp/nvflare/chapter2/kmeans/kmeans_iris`. Five rounds include the initialization round and four update rounds.

Continue with [survival analysis](../02.4.3_convert_survival_analysis_to_federated_learning/convert_survival_analysis_to_fl.ipynb).